# 04 — Train adv-NO on dynamic_dt export

Trains on data from [`01_export_dynamic_dt.ipynb`](01_export_dynamic_dt.ipynb).

| Knob | Description |
|------|-------------|
| `RUN_ALL_DT` | **`True`** → train all dt in `DT_DAYS_LIST` |
| `DT_DAYS` | Single dt when `RUN_ALL_DT=False` |
| `EXPORT_KEY` | `magicwin_reference` or `smallfixwin` (both uv, iD=2) |
| `GPU_POOL` | **`(6, 7, 0, 5, 1)`** — one dedicated cuda per dt |
| `EPOCHS` | **200** |
| `OPTIMIZER` | **adamw** (`WEIGHT_DECAY=1e-3`) |
| `VAL_SPLIT_MODE` | **`export`** — use train/val files from 01 |

**Current batch:** `DT_DAYS_LIST = [3, 23, 25, 27, 30]` → dt=3→cuda:6 · dt=23→cuda:7 · dt=25→cuda:0 · dt=27→cuda:5 · dt=30→cuda:1

**Original 8 dt (cuda:0–7):** commented in cell 1 — restore for `3,5,7,10,12,15,17,20`.

Run [`05_evaluate_rollout_dynamic_dt.ipynb`](05_evaluate_rollout_dynamic_dt.ipynb) after training.

In [1]:
import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MW_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "dynamic_dt" else (
    NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Magic_window" else NOTEBOOK_DIR / "Magic_window"
)
DT_DIR = MW_DIR / "dynamic_dt"
ROOT = MW_DIR.parent
for _p in (ROOT, MW_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import dynamic_dt.multi_dt_runner as _mdr
import dynamic_dt.train_runner as _tr
importlib.reload(_mdr)
importlib.reload(_tr)
from dynamic_dt.device_plan import (
    GPU_POOL_1_7,
    MID_DT_DAYS_LIST,
    configure_device_plan,
    format_dt_device_plan,
)
from dynamic_dt.dt_config import dt_hours
from dynamic_dt.window_catalog import get_window_spec, primary_export_component
from dynamic_dt.window_paths import train_prefix, val_prefix
from dynamic_dt.train_runner import build_train_config, prepare_train_job, train_one_dt
from dynamic_dt.multi_dt_runner import launch_train_all, print_job_summary
from src.train.adv_no_train import AdvNOTrainConfig, run_adv_no_training

MODELS_DIR = DT_DIR / "Models" / "adv_no"
LOG_DIR = DT_DIR / "logs"
for _d in (DT_DIR / "data", MODELS_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ============ USER KNOBS ============
EXPORT_KEY = "magicwin_reference"   # or "smallfixwin"

# --- Original 8 dt on cuda:0–7 (comment out mid-range block below to restore) ---
# from dynamic_dt.device_plan import STANDARD_DT_DAYS, clear_device_plan
# DT_DAYS_LIST = list(STANDARD_DT_DAYS)   # 3,5,7,10,12,15,17,20
# GPU_POOL = tuple(range(8))              # cuda:0 … cuda:7
# clear_device_plan()

# --- Custom dt: 3, 23, 25, 27, 30 — one dedicated GPU each (dt=3 re-added: prior model was overwritten by an aborted run) ---
DT_DAYS_LIST = [3, 23, 25, 27, 30]
GPU_POOL = (6, 7, 0, 5, 1)               # sorted dt→gpu: 3→cuda:6 · 23→cuda:7 · 25→cuda:0 · 27→cuda:5 · 30→cuda:1

RUN_ALL_DT = True                       # all dt in DT_DAYS_LIST
DT_DAYS = 25                            # only when RUN_ALL_DT=False

MAX_SAMPLES = None
EPOCHS = 200
OPTIMIZER = "adamw"
WEIGHT_DECAY = 1e-3
VAL_SPLIT = 0.3
VAL_SPLIT_MODE = "export"
VAL_SPLIT_SEED = 42
VAL_RECONSTRUCTION_METRIC = "l1"
DETACHED = True
RESUME = False                          # fresh run: clears prior artifacts for each dt before training
# ====================================

configure_device_plan(DT_DAYS_LIST, GPU_POOL)

if not RUN_ALL_DT and DT_DAYS not in DT_DAYS_LIST:
    raise ValueError(f"DT_DAYS must be one of {DT_DAYS_LIST}")

WINDOW = get_window_spec(EXPORT_KEY)
EXPORT_COMPONENT = primary_export_component(EXPORT_KEY)
train_kw = dict(
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    max_samples=MAX_SAMPLES,
    val_split=VAL_SPLIT,
    val_split_mode=VAL_SPLIT_MODE,
    val_split_seed=VAL_SPLIT_SEED,
    val_reconstruction_metric=VAL_RECONSTRUCTION_METRIC,
)

print(f"EXPORT_KEY={EXPORT_KEY} ({WINDOW['label']}) | RUN_ALL_DT={RUN_ALL_DT}")
print(f"GPU_POOL={GPU_POOL} | DT_DAYS_LIST={DT_DAYS_LIST}")
print(f"Optimizer: {OPTIMIZER}, lr=2e-4, weight_decay={WEIGHT_DECAY}, epochs={EPOCHS}")
print(f"val split: export (train/val prefixes from 01_export)")
print(format_dt_device_plan(DT_DAYS_LIST if RUN_ALL_DT else [DT_DAYS]))

EXPORT_KEY=magicwin_reference (Magic window (reference)) | RUN_ALL_DT=True
GPU_POOL=(6, 7, 0, 5, 1) | DT_DAYS_LIST=[3, 23, 25, 27, 30]
Optimizer: adamw, lr=2e-4, weight_decay=0.001, epochs=200
val split: export (train/val prefixes from 01_export)
  dt= 3d → cuda:6 (CUDA_VISIBLE_DEVICES=6)
  dt=23d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=25d → cuda:0 (CUDA_VISIBLE_DEVICES=0)
  dt=27d → cuda:5 (CUDA_VISIBLE_DEVICES=5)
  dt=30d → cuda:1 (CUDA_VISIBLE_DEVICES=1)


In [2]:
if RUN_ALL_DT:
    jobs = launch_train_all(
        dt_dir=DT_DIR,
        root=ROOT,
        export_key=EXPORT_KEY,
        log_dir=LOG_DIR,
        dt_days_list=DT_DAYS_LIST,
        detached=DETACHED,
        resume=RESUME,
        **train_kw,
    )
    print_job_summary(jobs, label="Train")
else:
    import numpy as np
    DT = dt_hours(DT_DAYS)
    data_prefix = train_prefix(DT_DIR, EXPORT_KEY, component=EXPORT_COMPONENT, dt_hours=DT)
    val_prefix_path = val_prefix(DT_DIR, EXPORT_KEY, component=EXPORT_COMPONENT, dt_hours=DT)
    for label, prefix in (("train", data_prefix), ("val", val_prefix_path)):
        p = Path(f"{prefix}_sims.npy")
        if not p.exists():
            raise FileNotFoundError(f"Missing {label} export: {p}\nRun 01_export_dynamic_dt.ipynb")
        print(f"{label}: {np.load(str(p), mmap_mode='r').shape}")

    if DETACHED:
        launched = train_one_dt(
            dt_dir=DT_DIR, root=ROOT, export_key=EXPORT_KEY, dt_days=DT_DAYS,
            log_dir=LOG_DIR, detached=True, resume=RESUME, **train_kw,
        )
        print(f"Started PID {launched['pid']} | tail -f {launched['log_path']}")
    else:
        job = prepare_train_job(dt_dir=DT_DIR, export_key=EXPORT_KEY, dt_days=DT_DAYS, **train_kw)
        config = AdvNOTrainConfig.from_dict(json.loads(Path(job["config_path"]).read_text()))
        run_adv_no_training(config, resume=RESUME)

Train plan:
  dt= 3d → cuda:6 (CUDA_VISIBLE_DEVICES=6)
  dt=23d → cuda:7 (CUDA_VISIBLE_DEVICES=7)
  dt=25d → cuda:0 (CUDA_VISIBLE_DEVICES=0)
  dt=27d → cuda:5 (CUDA_VISIBLE_DEVICES=5)
  dt=30d → cuda:1 (CUDA_VISIBLE_DEVICES=1)

--- train dt=3d ---

--- train dt=23d ---

--- train dt=25d ---



--- train dt=27d ---

--- train dt=30d ---

Train — 5 job(s):
  dt=3d gpu=6 PID 3913307 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/train_magicwin_reference_dt3d_gpu6.log
  dt=23d gpu=7 PID 3913437 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/train_magicwin_reference_dt23d_gpu7.log
  dt=25d gpu=0 PID 3913501 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/train_magicwin_reference_dt25d_gpu0.log
  dt=27d gpu=5 PID 3913565 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/train_magicwin_reference_dt27d_gpu5.log
  dt=30d gpu=1 PID 3913629 | tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Magic_window/dynamic_dt/logs/train_magicwin_reference_dt30d_gpu1.log
